In [ ]:
# %%
# Experimento: NTv3 8M (Fine-tuning COMPLETO) + Classificador (Arquitetura GLARE)
# Entrada: arquivos .fa por classe em  <DATA_ROOT>/<dataset>/Train/  e  .../Test/
import os
import re
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoConfig, AutoModelForMaskedLM

# =============================================
# CONFIGURAÇÃO
# =============================================
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o dispositivo: {device}")

dataset = 'Dataset3'
DATA_ROOT = './Data'
TRAIN_DIR = os.path.join(DATA_ROOT, dataset, 'Train')
TEST_DIR = os.path.join(DATA_ROOT, dataset, 'Test')

MODEL_ID = "InstaDeepAI/NTv3_8M_pre"  # Modelo menor de 8M
WIN = 4096
STRIDE = 2048

BATCH_SIZE = 8
NUM_EPOCHS = 10
PATIENCE = 5
LEARNING_RATE_BACKBONE = 5e-5
LEARNING_RATE_HEAD = 3e-4

ORDEM_EXIBICAO = None

# =============================================
# 1. ARQUITETURA DO MODELO (TIPO GLARE)
# =============================================
class GLAREHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 512, num_superfamilies: int = 45, dropout: float = 0.1):
        super().__init__()

        # Ajuste dinâmico: se a dimensão de entrada (256) for menor que o hidden_dim desejado (512),
        # nivelamos o hidden_dim para não forçar uma expansão artificial abrupta no MLP.
        if input_dim < hidden_dim:
            hidden_dim = input_dim

        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.superfamily_head = nn.Linear(hidden_dim // 2, num_superfamilies)

    def forward(self, x):
        h = self.trunk(x)
        return self.superfamily_head(h)


class GLAREModel(nn.Module):
    def __init__(self, base_model_name: str, num_classes: int):
        super().__init__()

        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
        config = AutoConfig.from_pretrained(base_model_name, trust_remote_code=True)

        config.stem_compute_dtype = "float32"
        config.down_convolution_compute_dtype = "float32"
        config.up_convolution_compute_dtype = "float32"
        config.core_compute_dtype = "float32"

        masked_lm = AutoModelForMaskedLM.from_pretrained(base_model_name, config=config, trust_remote_code=True)

        # Fine-tuning COMPLETO
        self.backbone = masked_lm
        for param in self.backbone.parameters():
            param.requires_grad = True

        # Correção da dimensão de entrada: o NTv3 usa 'd_model', mas inspecionamos diretamente a config
        hidden_size = getattr(config, "d_model", 256) # 256 é o padrão para o 8M

        self.head = GLAREHead(input_dim=hidden_size, hidden_dim=512, num_superfamilies=num_classes)

    def mean_pool(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).float()
        summed = (hidden_states * mask).sum(dim=1)
        count = mask.sum(dim=1).clamp(min=1)
        return summed / count

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        last_hidden = outputs.hidden_states[-1]
        pooled = self.mean_pool(last_hidden, attention_mask)
        logits = self.head(pooled)
        return logits


# =============================================
# 2. LEITURA, DATASET E PRE-PROCESSAMENTO
# =============================================
EXTS = ('.fa', '.fasta', '.fna')

def ler_fasta(path):
    ids, seqs = [], []
    cur_id, cur = None, []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line: continue
            if line.startswith('>'):
                if cur_id is not None:
                    ids.append(cur_id)
                    seqs.append(''.join(cur))
                cur_id, cur = line[1:].split()[0], []
            else:
                cur.append(line)
    if cur_id is not None:
        ids.append(cur_id)
        seqs.append(''.join(cur))
    return ids, seqs

def limpar(seq):
    return re.sub(r'[^ACGTN]', 'N', seq.upper().replace('U', 'T'))

def carregar_split(pasta, classes=None):
    arquivos = sorted(f for f in os.listdir(pasta) if f.lower().endswith(EXTS))
    nomes = [os.path.splitext(f)[0] for f in arquivos]
    if classes is None: classes = nomes
    ids, seqs, labels = [], [], []
    for f, nome in zip(arquivos, nomes):
        i, s = ler_fasta(os.path.join(pasta, f))
        ids += i
        seqs += [limpar(x) for x in s]
        labels += [classes.index(nome)] * len(s)
    return ids, seqs, np.array(labels, dtype=np.int64), classes

ids_tr, seqs_tr_all, y_tr_all, classes = carregar_split(TRAIN_DIR)
ids_te, seqs_te, y_te, _ = carregar_split(TEST_DIR, classes)
num_classes = len(classes)

complement_map = str.maketrans('ACGT', 'TGCA')
def reverse_complement(seq):
    return seq[::-1].translate(complement_map)

idx_all = np.arange(len(y_tr_all))
idx_tr, idx_val = train_test_split(idx_all, test_size=0.2, stratify=y_tr_all, random_state=42)
seqs_tr = [seqs_tr_all[i] for i in idx_tr]
y_tr = y_tr_all[idx_tr]
seqs_val = [seqs_tr_all[i] for i in idx_val]
y_val = y_tr_all[idx_val]

class TEDataset(Dataset):
    def __init__(self, seqs, labels, tokenizer, train_mode=False):
        self.seqs = seqs
        self.labels = labels
        self.tokenizer = tokenizer
        self.train_mode = train_mode
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = self.seqs[idx]
        label = self.labels[idx]

        if self.train_mode and np.random.rand() > 0.5:
            seq = reverse_complement(seq)

        if len(seq) > WIN:
            if self.train_mode:
                start = np.random.randint(0, len(seq) - WIN)
            else:
                start = 0
            seq = seq[start:start + WIN]

        encoded = self.tokenizer(seq, add_special_tokens=False, max_length=WIN, truncation=True, padding='max_length', return_tensors='pt')

        input_ids = encoded['input_ids'].squeeze(0)

        if 'attention_mask' in encoded:
            attention_mask = encoded['attention_mask'].squeeze(0)
        else:
            attention_mask = (input_ids != self.pad_token_id).long()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.long)
        }

tokenizer_temp = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
train_dataset = TEDataset(seqs_tr, y_tr, tokenizer_temp, train_mode=True)
val_dataset = TEDataset(seqs_val, y_val, tokenizer_temp, train_mode=False)
test_dataset = TEDataset(seqs_te, y_te, tokenizer_temp, train_mode=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# =============================================
# 3. TREINAMENTO
# =============================================
model = GLAREModel(base_model_name=MODEL_ID, num_classes=num_classes).to(device)

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': LEARNING_RATE_BACKBONE},
    {'params': model.head.parameters(), 'lr': LEARNING_RATE_HEAD}
], weight_decay=0.01)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

best_val_loss = float('inf')
epochs_no_improve = 0
best_model_path = "glare_ntv3_8m_finetuned.pt"

print("\nIniciando Fine-Tuning Completo...")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss, vcorrect, vtotal = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)

            val_loss += loss.item()
            vcorrect += (logits.argmax(dim=-1) == labels).sum().item()
            vtotal += labels.size(0)

    val_acc = vcorrect / vtotal
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Tempo: {time.time()-t0:.1f}s | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print("Early stopping acionado.")
            break

# =============================================
# 4. AVALIAÇÃO NO TESTE E MATRIZ DE CONFUSÃO
# =============================================
print("\nAvaliando no conjunto de Teste...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(input_ids, attention_mask)

        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

if ORDEM_EXIBICAO is not None:
    ordem_idx = [classes.index(c) for c in ORDEM_EXIBICAO]
else:
    ordem_idx = list(range(num_classes))
nomes_exib = [classes[i] for i in ordem_idx]

print("\nRelatório de Classificação (Teste):")
print(classification_report(all_labels, all_preds, target_names=nomes_exib, zero_division=0))
print(f"F1 ponderado final: {f1_score(all_labels, all_preds, average='weighted'):.4f}")

# Plotagem da Matriz de Confusão
cm = confusion_matrix(all_labels, all_preds, labels=ordem_idx)

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_xlabel('Rótulo Previsto', fontsize=18)
ax.set_ylabel('Rótulo Verdadeiro', fontsize=18)
ax.set_xticks(np.arange(len(nomes_exib)))
ax.set_yticks(np.arange(len(nomes_exib)))
ax.set_xticklabels(nomes_exib, fontsize=14)
ax.set_yticklabels(nomes_exib, fontsize=14)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

for t in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, t, f"{int(cm[t, j])}", ha="center", va="center",
                color="white" if t == j else "black", fontsize=18)

cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.tick_params(labelsize=14)
plt.tight_layout()
plt.show()

Usando o dispositivo: cuda


Loading weights: 100%|██████████| 147/147 [00:00<00:00, 2734.98it/s]



Iniciando Fine-Tuning Completo...
Epoch 1/10 | Tempo: 1096.7s | Train Loss: 1.1539 | Train Acc: 0.6916 | Val Loss: 1.0415 | Val Acc: 0.7417
Epoch 2/10 | Tempo: 1057.2s | Train Loss: 0.8821 | Train Acc: 0.7941 | Val Loss: 0.9040 | Val Acc: 0.7846
Epoch 3/10 | Tempo: 1025.9s | Train Loss: 0.7965 | Train Acc: 0.8264 | Val Loss: 0.7873 | Val Acc: 0.8328
Epoch 4/10 | Tempo: 1025.3s | Train Loss: 0.7409 | Train Acc: 0.8525 | Val Loss: 0.7236 | Val Acc: 0.8539
Epoch 5/10 | Tempo: 1026.2s | Train Loss: 0.6982 | Train Acc: 0.8687 | Val Loss: 0.7370 | Val Acc: 0.8551
Epoch 6/10 | Tempo: 1026.3s | Train Loss: 0.6549 | Train Acc: 0.8846 | Val Loss: 0.7970 | Val Acc: 0.8505
Epoch 7/10 | Tempo: 1026.4s | Train Loss: 0.6202 | Train Acc: 0.8991 | Val Loss: 0.7434 | Val Acc: 0.8708
Epoch 8/10 | Tempo: 1024.7s | Train Loss: 0.5974 | Train Acc: 0.9086 | Val Loss: 0.7699 | Val Acc: 0.8611
